<h1>SQL Practice with NJ Incarcerated Population Census Data<h1>

<h3>1. Importing SQLite and Pandas<h3>

In [89]:
import pandas as pd
import sqlite3

<h3>1.1. Loading SQLite and connecting to the practice database<h3>

In [ ]:
%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [102]:
%sql sqlite:///practice_database_census_v2.db

<h3>2. Querying, cleaning, and ranking data using CTEs and window functions<h3>

<h3>2.1. Categorizing and ranking respondents by age and county court.<h3>

In [ ]:
%%sql
-- Respondent ages are categorized into 1 of 4 categories.

with categories as (select id, case
when age between 19 and 24 then '19_24'
when age between 25 and 34 then '25_34'
when age between 35 and 44 then '35_44'
else '45+' end as age_cats
from Census
),

-- A window function is used to count the number of ids for each age category.

cleaned_data as(
select age_cats, count(id) over(partition by age_cats) as count_cats
from categories
)

/* 
The number of respondents by age category is shown by pulling the 
result from the previous CTE and grouping it by age category.
*/

select age_cats, count_cats
from cleaned_data
group by age_cats;

In [ ]:
%%sql
-- Creating a table of the number of respondents for each unique county court.

with rank as(
select county, count(id) as count_id
from Census
group by county
order by count(id) asc
)

/* 
Using window functions to add rank and dense rank columns in order to rank the county courts by count of respondents.
If multiple county courts fall in the same rank then the next rank is skipped. No ranks are skipped when 
using dense rank.
*/

select county, count_id, rank() over(order by count_id) as rank, dense_rank() over(order by count_id) as dense_rank
from rank;

<h3>2.2. Cleaning dates for comparison and analysis<h3>

In [ ]:
%%sql

/* 
Since the admit and release dates are not in the optimal format for querying, some extra cleaning steps are required.
The query below starts this process by returning the location of the "/" characters within the admit and release dates 
for each row.
*/

select admit_date, release_date, 
instr(admit_date, '/') as admit_first_dash, 
instr(substr(admit_date, instr(admit_date, '/') + 1),'/') + instr(admit_date, '/') as admit_sec_dash,
instr(release_date, '/') as rel_first_dash, 
instr(substr(release_date, instr(release_date, '/') + 1),'/') + instr(release_date, '/') as rel_sec_dash
from Dates;

In [ ]:
%%sql

/* 
The logic from the previous query is used to seperate the date components
based on their location relative to the nearest "/" symbol.
*/

select admit_date, release_date, substr(admit_date, instr(admit_date, '/'), - 2) as admit_month,
substr(admit_date, instr(admit_date, '/') + 1, + 2) as admit_day,
'20' || substr(admit_date, instr(substr(admit_date, instr(admit_date, '/') + 1), '/') + 1 + instr(admit_date, '/'), 2) as admit_year,
substr(release_date, instr(release_date, '/'), - 2) as rel_month,
substr(release_date, instr(release_date, '/') + 1, + 2) as rel_day,
'20' || substr(release_date, instr(substr(release_date, instr(release_date, '/') + 1), '/') + 1 + instr(release_date, '/'), 2) as rel_year
from Dates;

In [ ]:
%%sql

/* 
With the date components selected, additional cleaning is preformed (all "/" symbols are removed, 
all day and month values are made to be 2 digits, and "20" is added to the year component of each date). 
By wrapping the previous query into a CTE, the logic steps are easier to display.
*/

with cols as (
select admit_date, release_date, substr(admit_date, instr(admit_date, '/'), - 2) as admit_month,
substr(admit_date, instr(admit_date, '/') + 1, + 2) as admit_day,
'20' || substr(admit_date, instr(substr(admit_date, instr(admit_date, '/') + 1), '/') + 1 + instr(admit_date, '/'), 2) as admit_year,
substr(release_date, instr(release_date, '/'), - 2) as rel_month,
substr(release_date, instr(release_date, '/') + 1, + 2) as rel_day,
'20' || substr(release_date, instr(substr(release_date, instr(release_date, '/') + 1), '/') + 1 + instr(release_date, '/'), 2) as rel_year
from Dates
)

select admit_date, release_date, substr('00' || admit_month, -2) as clean_adm_month, substr('00' || cast(admit_day as int), -2) as clean_adm_day, admit_year,
substr('00' || rel_month, -2) as clean_rel_month, substr('00' || cast(rel_day as int), -2) as clean_rel_day, rel_year
from cols;

In [ ]:
%%sql

/* 
All logic from the last 3 queries are combined too preform the entire cleaning process. The resulting dates are in the 
acceptable SQL formate of YYYY-MM-DD.
*/

with dash_loc as (
select admit_date, release_date, 
instr(admit_date, '/') as admit_first_dash, 
instr(substr(admit_date, instr(admit_date, '/') + 1),'/') + instr(admit_date, '/') as admit_sec_dash,
instr(release_date, '/') as rel_first_dash, 
instr(substr(release_date, instr(release_date, '/') + 1),'/') + instr(release_date, '/') as rel_sec_dash
from Dates
),

cols as (
select admit_date, release_date, substr(admit_date, instr(admit_date, '/'), - 2) as admit_month,
substr(admit_date, instr(admit_date, '/') + 1, + 2) as admit_day,
'20' || substr(admit_date, instr(substr(admit_date, instr(admit_date, '/') + 1), '/') + 1 + instr(admit_date, '/'), 2) as admit_year,
substr(release_date, instr(release_date, '/'), - 2) as rel_month,
substr(release_date, instr(release_date, '/') + 1, + 2) as rel_day,
'20' || substr(release_date, instr(substr(release_date, instr(release_date, '/') + 1), '/') + 1 + instr(release_date, '/'), 2) as rel_year
from Dates
),

cleaned_cols as (
select admit_date, release_date, substr('00' || admit_month, -2) as clean_adm_month, substr('00' || cast(admit_day as int), -2) as clean_adm_day, admit_year,
substr('00' || rel_month, -2) as clean_rel_month, substr('00' || cast(rel_day as int), -2) as clean_rel_day, rel_year
from cols
)

select admit_date, release_date, admit_year || '-' || clean_adm_month || '-' || clean_adm_day as clean_adm_date,
rel_year || '-' || clean_rel_month || '-' || clean_rel_day as clean_rel_date
from cleaned_cols;


In [ ]:
%%sql

/* 
Additional logic is added to calculate the difference between the admit and release dates. You can see 
that when the same logic is applied to the uncleaned dates (unclean_timediff), the query returns "None".
*/

with dash_loc as (
select admit_date, release_date, 
instr(admit_date, '/') as admit_first_dash, 
instr(substr(admit_date, instr(admit_date, '/') + 1),'/') + instr(admit_date, '/') as admit_sec_dash,
instr(release_date, '/') as rel_first_dash, 
instr(substr(release_date, instr(release_date, '/') + 1),'/') + instr(release_date, '/') as rel_sec_dash
from Dates
),

cols as (
select admit_date, release_date, substr(admit_date, instr(admit_date, '/'), - 2) as admit_month,
substr(admit_date, instr(admit_date, '/') + 1, + 2) as admit_day,
'20' || substr(admit_date, instr(substr(admit_date, instr(admit_date, '/') + 1), '/') + 1 + instr(admit_date, '/'), 2) as admit_year,
substr(release_date, instr(release_date, '/'), - 2) as rel_month,
substr(release_date, instr(release_date, '/') + 1, + 2) as rel_day,
'20' || substr(release_date, instr(substr(release_date, instr(release_date, '/') + 1), '/') + 1 + instr(release_date, '/'), 2) as rel_year
from Dates
),

cleaned_cols as (
select admit_date, release_date, substr('00' || admit_month, -2) as clean_adm_month, substr('00' || cast(admit_day as int), -2) as clean_adm_day, admit_year,
substr('00' || rel_month, -2) as clean_rel_month, substr('00' || cast(rel_day as int), -2) as clean_rel_day, rel_year
from cols
),

final_dates as (
select admit_date, release_date, admit_year || '-' || clean_adm_month || '-' || clean_adm_day as clean_adm_date,
rel_year || '-' || clean_rel_month || '-' || clean_rel_day as clean_rel_date
from cleaned_cols
)

-- Added Lag and Lead window functions to display the cleaned admission date value of the next and previous rows.

select clean_adm_date, clean_rel_date, timediff(clean_rel_date, clean_adm_date) as cleaned_timediff,
admit_date, release_date, timediff(release_date, admit_date) as unclean_timediff, lead(clean_adm_date, 1) over(order by clean_adm_date) as Lead,
lag (clean_adm_date, 1) over(order by clean_adm_date) as Lag
from final_dates;